# Mounting Google Drive

In [3]:
# Mounting google drive to access images
from google.colab import drive
drive.mount('/content/drive')

# Load the (desired) directory
%cd drive/MyDrive/workshop

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/workshop


In [5]:
%cd I2K_workshop

/content/drive/.shortcut-targets-by-id/1LR7LS5NVXZ3crsj1eu0pk7S-4MjyXXkI/I2K_workshop


# Install Gradio

In [6]:
# install gradio
!pip install gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.6/50.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.7/56.7 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.8/319.8 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.0/78.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.5/447.5 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.3/73.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/13

In [ ]:
# load gradio in jupyter notebook
%load_ext gradio

# Example

In [14]:
import gradio as gr
import numpy as np


stack_dir = "examples/"

def sepia(input_img):
    sepia_filter = np.array([
        [0.393, 0.769, 0.189],
        [0.349, 0.686, 0.168],
        [0.272, 0.534, 0.131]
    ])
    sepia_img = input_img.dot(sepia_filter.T)
    sepia_img /= sepia_img.max()
    return sepia_img

demo = gr.Interface(sepia, gr.Image(), "image")


demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://828676c05cdbf64c49.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Visualizing results with Gradio

In [25]:
# @title
# build UNet
import tensorflow as tf
import tensorflow.keras.layers as layers
import tensorflow.keras.models as models
import tensorflow.keras.metrics as metrics
#from keras import backend as keras


def unet(pretrained_weights = None, input_size = (256,256,1)):
    inputs = layers.Input(input_size)
    conv1 = layers.Conv2D(64, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(inputs)
    conv1 = layers.Conv2D(64, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv1)
    pool1 = layers.MaxPooling2D(pool_size=(2, 2))(conv1)

    conv2 = layers.Conv2D(128, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(pool1)
    conv2 = layers.Conv2D(128, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv2)
    pool2 = layers.MaxPooling2D(pool_size=(2, 2))(conv2)

    conv3 = layers.Conv2D(256, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(pool2)
    conv3 = layers.Conv2D(256, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv3)
    pool3 = layers.MaxPooling2D(pool_size=(2, 2))(conv3)

    conv4 = layers.Conv2D(512, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(pool3)
    conv4 = layers.Conv2D(512, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv4)
    drop4 = layers.Dropout(0.5)(conv4)
    pool4 = layers.MaxPooling2D(pool_size=(2, 2))(drop4)

    conv5 = layers.Conv2D(1024, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(pool4)
    conv5 = layers.Conv2D(1024, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv5)
    drop5 = layers.Dropout(0.5)(conv5)

    up6 = layers.Conv2D(512, 2, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(layers.UpSampling2D(size = (2,2))(drop5))
    merge6 = layers.concatenate([drop4,up6], axis = 3)
    conv6 = layers.Conv2D(512, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(merge6)
    conv6 = layers.Conv2D(512, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv6)

    up7 = layers.Conv2D(256, 2, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(layers.UpSampling2D(size = (2,2))(conv6))
    merge7 = layers.concatenate([conv3,up7], axis = 3)
    conv7 = layers.Conv2D(256, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(merge7)
    conv7 = layers.Conv2D(256, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv7)

    up8 = layers.Conv2D(128, 2, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(layers.UpSampling2D(size = (2,2))(conv7))
    merge8 = layers.concatenate([conv2,up8], axis = 3)
    conv8 = layers.Conv2D(128, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(merge8)
    conv8 = layers.Conv2D(128, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv8)

    up9 = layers.Conv2D(64, 2, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(layers.UpSampling2D(size = (2,2))(conv8))
    merge9 = layers.concatenate([conv1,up9], axis = 3)
    conv9 = layers.Conv2D(64, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(merge9)
    conv9 = layers.Conv2D(64, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv9)
    conv9 = layers.Conv2D(2, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv9)

    conv10 = layers.Conv2D(1, 1, activation = 'sigmoid')(conv9)

    model = models.Model(inputs=inputs, outputs=conv10)

    if(pretrained_weights):
        model.load_weights(pretrained_weights)

    return model

In [28]:
import os
import numpy as np
import skimage.transform as trans
from skimage.color import rgb2gray


def predict_model(input):
    # need to change this path
    model_path = "trained_models/"
    h, w = 256, 256
    input_shape = [h, w, 1]
    output_channels = 1
    batch_size = 1

    # convert image into numpy array and reshape it into model's input size
    img = trans.resize(input, (w, h))
    result_img = img.copy()
    img = rgb2gray(img).reshape(1, h, w, 1)

    # load the pretrained model
    model_name = "unetv0_sgd500_neptune"
    model_file = os.path.join(model_path, model_name + ".hdf5")
    model = unet(model_file)

    # Predict and save the results as numpy array
    results = model.predict(img)

    pred = np.copy(results)
    pred[pred >= 0.5] = 1
    pred[pred < 0.5] = 0
    output =  np.array(pred[0][:,:,0])

    # visualize the output mask
    seg_color = [0, 0, 255]
    masked = output != 0
    result_img[masked] = seg_color

    return result_img



In [27]:
import gradio as gr
import numpy as np
import glob
#from predict_unet import predict_model

title = "<center><strong><font size='8'> Medical Image Segmentation with UNet </font></strong></center>"

# need to add example images to the folder
sample_img = glob.glob("examples/*")
examples = []
for img in sample_img:
    examples.append([img])

def run_unet(input):
    output = predict_model(input)
    normalized_output = np.clip(output, 0, 1)
    return normalized_output
    #return 1


input_img = gr.Image(label="Input", type='numpy')
segm_img = gr.Image(label="Segmented Image")


with gr.Blocks(title='UNet examples') as demo:
    # build input and output for running UNet
    with gr.Tab("UNet"):
        # display input image and segmented image
        with gr.Row(variant="panel"):
            with gr.Column(scale=1):
                input_img.render()

            with gr.Column(scale=1):
                segm_img.render()

        # submit and clear
        with gr.Row():
            with gr.Column():
                segment_btn = gr.Button("Run Segmentation", variant='primary')
                clear_btn = gr.Button("Clear", variant="secondary")

                # load examples
                gr.Markdown("Try some of the examples below")
                gr.Examples(examples=examples,
                            inputs=[input_img],
                            outputs=segm_img,
                            fn=run_unet,
                            cache_examples=False,
                            examples_per_page=5)

            # just a placeholder for second column
            with gr.Column():
                gr.Markdown("")

        segment_btn.click(run_unet,
                            inputs=[
                                input_img,
                            ],
                            outputs=segm_img)



    def clear():
        return None, None

    clear_btn.click(clear, outputs=[input_img, segm_img])

demo.queue()
demo.launch(share=True)




Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://706c3d2ea1f21fbb53.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Resources for further learning

### UNet paper: https://arxiv.org/abs/1505.04597

### Neptune.ai: https://docs.neptune.ai/about/intro

### Gradio: https://www.gradio.app/main/docs/gradio/interface

### HuggingFace: https://huggingface.co/docs